# TP - Python - Flask
---

## Etapa 1

In [17]:
%%file tp.py
from flask import Flask, jsonify
from flasgger import Swagger

app = Flask(__name__)
swagger = Swagger(app)

productos = [
    {'id':1, 'nombre':'python', 'precio':10.0},
    {'id':2, 'nombre':'javascript', 'precio':12.0},
    {'id':3, 'nombre':'java', 'precio':19.0}
]

carrito = {}

@app.get('/')
def get_productos():
    """
    Obtener lista de productos
    ---
    responses:
      200:
        description: Lista de productos disponibles
        schema:
          type: array
          items:
            type: object
            properties:
              id:
                type: integer
              nombre:
                type: string
              precio:
                type: number
    """
    return jsonify(productos)

@app.get('/carrito')
def get_carrito():
    """
    Obtener contenido del carrito
    ---
    responses:
      200:
        description: Carrito actual con cantidades
        schema:
          type: object
    """
    return jsonify(carrito)

@app.post('/agregar/<int:id>')
def agregar(id):
    """
    Agregar producto al carrito
    ---
    parameters:
      - name: id
        in: path
        type: integer
        required: true
        description: ID del producto a agregar
    responses:
      200:
        description: Carrito actualizado
    """
    prod = next((p for p in productos if p['id'] == id), None)
    if prod is None:
        return jsonify({'error': 'Producto no encontrado'}), 404
    carrito[id] = carrito.get(id, 0) + 1
    return jsonify(carrito)

@app.delete('/eliminar/<int:id>')
def eliminar(id):
    """
    Eliminar producto del carrito
    ---
    parameters:
      - name: id
        in: path
        type: integer
        required: true
        description: ID del producto a eliminar
    responses:
      200:
        description: Carrito actualizado
    """
    if id not in carrito:
        return jsonify({'error': 'Producto no está en el carrito'}), 404
    if carrito[id] > 1:
        carrito[id] -= 1
    else:
        del carrito[id] 
    return jsonify(carrito)

@app.get('/total')
def total():
    """
    Calcular total de la compra
    ---
    responses:
      200:
        description: Total acumulado del carrito
        schema:
          type: object
          properties:
            total:
              type: number
    """
    total = 0.0
    for id, cantidad in carrito.items():
        producto = next((p for p in productos if p['id'] == id), None)
        if producto:
            total += producto['precio'] * cantidad   
    return jsonify({'total': total})

@app.delete('/vaciar')
def vaciar_carrito():
    """
    Vaciar el carrito
    ---
    responses:
      200:
        description: Carrito vaciado correctamente
        schema:
          type: object
          properties:
            mensaje:
              type: string
            carrito:
              type: object
    """
    carrito.clear()
    return jsonify({'mensaje': 'Carrito vaciado', 'carrito': carrito})

Overwriting tp.py


In [2]:
!export FLASK_APP=tp.py

In [3]:
!flask --app tp run

 * Serving Flask app 'tp'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
^C


[**swagger**: http://127.0.0.1:5000/apidocs/](http://127.0.0.1:5000/apidocs/)

In [20]:
import unittest
import requests

class TestAPIs(unittest.TestCase):

    URL_BASE = 'http://127.0.0.1:5000'

    def setUp(self):
        requests.delete(f'{self.URL_BASE}/vaciar')
        
    def test_status_code_ok(self):
        response = requests.get(self.URL_BASE)
        self.assertEqual(response.status_code, 200)

    def test_cantidad_productos(self):
        response = requests.get(self.URL_BASE)
        self.assertEqual(len(response.json()), 3)

    def test_agregar_mismo_producto(self):
        requests.post(f'{self.URL_BASE}/agregar/1')
        requests.post(f'{self.URL_BASE}/agregar/1')
        response = requests.get(f'{self.URL_BASE}/carrito')
        self.assertEqual(len(response.json()), 1)

    def test_agregar_distintos_productos(self):
        requests.post(f'{self.URL_BASE}/agregar/1')
        requests.post(f'{self.URL_BASE}/agregar/2')
        response = requests.get(f'{self.URL_BASE}/carrito')
        self.assertEqual(len(response.json()), 2)

    def test_total(self):
        requests.post(f'{self.URL_BASE}/agregar/1')
        requests.post(f'{self.URL_BASE}/agregar/2')
        response = requests.get(f'{self.URL_BASE}/total')
        self.assertEqual(response.json()['total'], 22)

In [21]:
unittest.main(defaultTest='TestAPIs', argv=[''], verbosity=2, exit=False)

test_agregar_distintos_productos (__main__.TestAPIs.test_agregar_distintos_productos) ... ok
test_agregar_mismo_producto (__main__.TestAPIs.test_agregar_mismo_producto) ... ok
test_cantidad_productos (__main__.TestAPIs.test_cantidad_productos) ... ok
test_status_code_ok (__main__.TestAPIs.test_status_code_ok) ... ok
test_total (__main__.TestAPIs.test_total) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.069s

OK
